# Item 1 extraction and competition measurement

Adapted from the supplied original workflow. Outputs and personal paths have been removed. Read `docs/PREPARING_DATA.md` and configure `configs/preparation.local.json` first. Run cells in order. This adaptation has not been validated by a complete raw-to-neural rerun.


# Lean 10-K Competition Dataset

This self-contained Notebook retains the six originally requested fields, five identifiers for company/filing/paragraph/sentence joins, four paragraph text descriptors and one review flag. It processes ALL retained sentence units, including competition=0. A sentence row repeats its parent paragraph and report features.

Speed changes: CPU processes with bounded queues; reuse of compatible v2.1 extraction caches; compact low-compression caches; batch sentence splitting; cached repeated sentence sentiment; competition scoring only within the selected business section (with the original full-document previous-three-word context); no full-10K competition metric, per-occurrence keyword audit, XPath extraction, embedding code, duplicate positive worksheet or five normalized-table exports. XLSX uses fast ZIP compression. Less work and fewer repeated columns reduce time and disk use, but no speed multiplier is guaranteed.

A single shared LDA is fitted from a reproducible year/quarter-stratified source sample processed FIRST. Its topic definition stays fixed for every exported filing in this run. The first Excel part is finalized while the remaining reports are still being extracted. All selected sources are processed: the LDA training cap is NOT an extraction cap.

Output goes to the LOCAL `~/10K_competition_lean16` folder by default, reducing continuous Dropbox synchronization during computation. Input still comes from ROOT in Dropbox; mark input ZIPs available offline before a large run. Existing v2.1 files are read for reuse and never edited. This Notebook does not download missing years or add ROI, financial matching, disclosure dates, or embeddings.


## 1. Existing environment

Use the Jupyter kernel that successfully ran v2.1. No new dependency is required. If starting fresh, install the following once, restart the kernel and continue at step 2. PyArrow, PyTorch and sentence-transformers are not used.

The pinned CPU package set targets Python 3.9–3.12. Installation is commented out intentionally so an active environment is not modified automatically.


In [ ]:
# Run ONLY if these packages are not installed, then restart the kernel.
# %pip install "numpy==1.26.4" "pandas==2.2.3" "scipy==1.13.1" "scikit-learn==1.5.2" "spacy==3.7.5" "vaderSentiment==3.3.2" "ftfy==6.3.1" "lxml==5.3.0" "openpyxl==3.1.5" "joblib==1.4.2" "tqdm==4.67.1"


## Configuration

Raw annual ZIP files are read from `raw_filings_dir`; writes go to the separate configured output root. Coverage can be partial. Report IDs and corpus identity can change when raw paths or archive timestamps change. For exact existing results, use the frozen manifest and caches instead of rebuilding.


In [ ]:
from pathlib import Path
import os,re,json,gzip,hashlib,time,sys,platform,subprocess,signal,csv
from collections import Counter
from zipfile import ZipFile,is_zipfile
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from pathlib import Path
import sys
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "project_config.py").is_file()), None)
if REPO_ROOT is None:
    raise RuntimeError("Start Jupyter from the repository root or notebooks directory.")
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))
from preparation_config import load_preparation_config
PREP = load_preparation_config(REPO_ROOT)

ROOT = PREP["raw_filings_dir"]
OUTPUT_ROOT = PREP["extraction_output_root"]
RUN_SCOPE = 'all'                 # 'pilot', 'quarter', or 'all'
TEST_YEAR, TEST_QUARTER = 2010, 'QTR1'
YEAR_START, YEAR_END = 2010, 2026
PILOT_FILES = 100
ALLOW_PARTIAL_COLLECTION = True  # Process available years; coverage is recorded explicitly

# Two workers for smaller Macs, up to four for machines with >=12 GiB RAM.
def available_ram_gib():
    try:
        if sys.platform=='darwin':return int(subprocess.check_output(['sysctl','-n','hw.memsize']))/2**30
        return os.sysconf('SC_PAGE_SIZE')*os.sysconf('SC_PHYS_PAGES')/2**30
    except Exception:return 8.
CPU_WORKERS = min(4 if available_ram_gib()>=12 else 2, max(1,(os.cpu_count() or 2)-1))
# Set CPU_WORKERS=1 if your environment cannot start subprocess workers.

LDA_TRAIN_FILINGS = 300           # Reproducible year/quarter-stratified source sample, processed FIRST
LDA_MAX_TOPICS = 10               # Actual K=min(this limit, max(2, accepted training documents//50))
LDA_ITERATIONS = 5
LDA_MODEL_TO_REUSE = None         # Optional explicit path to a compatible v2 or lean joblib bundle
EXPORT_EXCEL = True
EXPORT_CSV = False               # Fastest export: False for Excel and True for CSV
EXCEL_ROWS_PER_FILE = 25000
EXPORT_BATCH_FILINGS = 32

LITE_VERSION = 'lean16_v1_20261024'
PARSER_VERSION = 'item1_combined_and_inline_headings_v2_1_20261022'
RULE_VERSION = 'fivewords_optional_s_prev3_global_alpha_tokens_v2'
CLEANER_VERSION = 'utf8_cp1252_c1_unicode_repair_v2'
COLS = ['paragraph_text','sentence_text','competition_binary','paragraph_competition_sentence_count',
        'item1_lda_topic_probabilities','item1_sentiment',
        'cik','fiscal_year','doc_id','paragraph_id','sentence_id',
        'para_word_count','para_avg_sentence_words','para_avg_word_chars','para_type_token_ratio','review_required']

if not ROOT.is_dir():raise FileNotFoundError(f'Edit ROOT: input folder does not exist: {ROOT}')
if RUN_SCOPE not in ('pilot','quarter','all'):raise ValueError('Invalid RUN_SCOPE')
if not (EXPORT_EXCEL or EXPORT_CSV):raise ValueError('Enable at least one export format')
if CPU_WORKERS<1 or LDA_TRAIN_FILINGS<5 or LDA_MAX_TOPICS<2:raise ValueError('Invalid worker/model settings')
if not 1<=EXCEL_ROWS_PER_FILE<=1048575:raise ValueError('Invalid Excel row limit')
run_name='all_years' if RUN_SCOPE=='all' else f'{TEST_YEAR}_{TEST_QUARTER}'
OUT=OUTPUT_ROOT/run_name
CACHE=OUT/'cache';MODEL_DIR=OUT/'model';EXPORT_DIR=OUT/'data';RUNTIME=OUT/'runtime'
for directory in (CACHE,MODEL_DIR,EXPORT_DIR,RUNTIME):directory.mkdir(parents=True,exist_ok=True)
LEGACY_CACHE_DIRS=[str(ROOT/'_competition_dataset_v2_1'/'all_years'/'cache'),
                   str(ROOT/'_competition_dataset_v2_1'/f'{TEST_YEAR}_{TEST_QUARTER}'/'cache')]


def stable_hash(value):return hashlib.sha256(value.encode('utf-8')).hexdigest()
def atom_json(path,value):
    path=Path(path);temporary=path.with_name(path.name+'.tmp')
    temporary.write_text(json.dumps(value,ensure_ascii=False,indent=2),encoding='utf-8');temporary.replace(path)
def read_record(ref):
    with gzip.open(ref['cache_path'],'rt',encoding='utf-8') as handle:return json.load(handle)
def item1_text(ref):return '\n\n'.join(p['paragraph_text'] for p in read_record(ref)['paragraphs'])

print('Output:',OUT)
print('Workers:',CPU_WORKERS,'| RAM GiB:',round(available_ram_gib(),1),'| main columns:',len(COLS))
print('Embedding: absent. Existing compatible v2.1 caches will be reused when found.')


## 3. Discover sources and prioritize the shared-model training sample

Supported sources are `ROOT/2010/QTR1/*.htm` and equivalent year/quarter directories, or `ROOT/10K_2010.zip` with quarter directories inside. Available ZIPs take precedence over extracted folders to avoid double counting. Apple metadata files are excluded. Source order is deterministic.

New lean cache fingerprints use source path/size/mtime and, for ZIP members, CRC and uncompressed size. Ordinary source changes invalidate the cache; do not manually replace a source while preserving all of its metadata. Existing v2.1 cache reuse additionally matches raw-content SHA-256, parser/rule version, NLP package fingerprint and the original cache key. Reuse requires the same package versions and default v2.1 override settings; incompatible caches are recomputed.


In [ ]:
def quarter_of(name):
    match=re.search(r'(?:^|[/\\])(QTR[1-4])(?:[/\\]|$)',str(name),re.I)
    return match.group(1).upper() if match else 'unknown'

def spec_key(spec):
    name=Path(spec['member'] or spec['file']).name;match=re.match(r'(\d+)_',name)
    return (int(spec['filing_year']),spec['quarter'],int(match.group(1)) if match else 10**15,name)

def discover():
    jobs=[];coverage=[]
    years=range(YEAR_START,YEAR_END+1) if RUN_SCOPE=='all' else [TEST_YEAR]
    for year in years:
        archive=ROOT/f'10K_{year}.zip';folder=ROOT/str(year);year_jobs=[]
        if archive.is_file() and is_zipfile(archive):
            stat=archive.stat();archive_stamp=f'{archive.resolve()}|{stat.st_size}|{stat.st_mtime_ns}'
            with ZipFile(archive) as reader:
                for info in reader.infolist():
                    name=info.filename
                    if not name.lower().endswith(('.htm','.html')) or '__MACOSX/' in name or Path(name).name.startswith('._'):continue
                    quarter=quarter_of(name)
                    if RUN_SCOPE!='all' and quarter!=TEST_QUARTER:continue
                    stamp=stable_hash(f'{archive_stamp}|{name}|{info.CRC}|{info.file_size}')
                    year_jobs.append({'file':str(archive),'member':name,'filing_year':str(year),'quarter':quarter,'stamp':stamp})
        elif folder.is_dir():
            search=folder if RUN_SCOPE=='all' else folder/TEST_QUARTER
            for path in search.rglob('*'):
                if not path.is_file() or path.suffix.lower() not in ('.htm','.html') or path.name.startswith('._') or '__MACOSX' in path.parts:continue
                stat=path.stat();stamp=stable_hash(f'{path.resolve()}|{stat.st_size}|{stat.st_mtime_ns}')
                year_jobs.append({'file':str(path),'member':'','filing_year':str(year),'quarter':quarter_of(path),'stamp':stamp})
        jobs.extend(year_jobs)
        for quarter in ('QTR1','QTR2','QTR3','QTR4','unknown'):
            coverage.append({'filing_year':year,'quarter':quarter,'available_sources':sum(j['quarter']==quarter for j in year_jobs)})
        if not year_jobs and not ALLOW_PARTIAL_COLLECTION:raise RuntimeError(f'No available input for {year}; finish download or explicitly permit partial coverage.')
    if not jobs:raise RuntimeError('No readable HTML inputs found. Check ROOT and Dropbox download state.')
    pd.DataFrame(coverage).to_csv(OUT/'coverage.csv',index=False)
    jobs=sorted(jobs,key=spec_key)
    if RUN_SCOPE=='pilot':jobs=jobs[:PILOT_FILES]
    return jobs


def training_first(jobs):
    if LDA_MODEL_TO_REUSE is not None:return jobs,0
    maximum=min(LDA_TRAIN_FILINGS,len(jobs));rng=np.random.default_rng(42);groups={}
    for i,spec in enumerate(jobs):groups.setdefault((spec['filing_year'],spec['quarter']),[]).append(i)
    selected=set();quota=max(1,maximum//max(1,len(groups)))
    for indexes in groups.values():selected.update(rng.choice(indexes,min(quota,len(indexes)),replace=False).tolist())
    if len(selected)>maximum:selected=set(rng.choice(sorted(selected),maximum,replace=False).tolist())
    left=sorted(set(range(len(jobs)))-selected)
    if len(selected)<maximum:selected.update(rng.choice(left,maximum-len(selected),replace=False).tolist())
    # Group archive reads while retaining a reproducible sample spanning available years/quarters.
    return [jobs[i] for i in sorted(selected)]+[job for i,job in enumerate(jobs) if i not in selected],len(selected)

jobs=discover();ordered_jobs,training_jobs=training_first(jobs)
print('Selected sources:',len(jobs),'| initial training sources:',training_jobs)
print('Check coverage.csv: unavailable years/quarters are not automatically downloaded.')


## 4. Load the CPU runner

The runner below is embedded in this Notebook and written automatically to the output runtime directory. It uses the SAME Python executable as Jupyter and a main-guarded spawn process pool, which avoids pickling Notebook-defined functions. Each process keeps its own ZIP reader. Only small filing summaries return to Jupyter; full compact records are saved per filing.

Competition uses exact case-insensitive tokens: competition, competitor, competitive, compete, competing, each also accepting a final s. Exclude a target when not/less/few/limited appears among its previous THREE full-document English word tokens; punctuation and block boundaries do not reset the window. Headings/tables still contribute to that context, but detected headings, personnel-table rows and non-narrative labels do not become sentence rows. Known inline heading prefixes are split from retained body text. Purely short text is not blanket-deleted.

Retained paragraphs use the same spaCy sentencizer and VADER unweighted mean over all sentence/list units as v2.1. LRU sentiment caching changes execution time only, not scores. No positive-only text filtering is performed.


In [ ]:
RUNNER_SOURCE = 'from pathlib import Path\nimport os, re, json, gzip, hashlib, time, shutil, platform, bisect, io, csv, sys\nfrom collections import Counter\nfrom datetime import datetime, timezone\nimport importlib.metadata as metadata\nimport numpy as np\nimport pandas as pd\nfrom lxml import html, etree\n\n\n\nMETADATA_OVERRIDES_CSV=None\n\nWORD_RE = re.compile(r\'[A-Za-z]+\')\nCOMP_WORDS = {word+suffix for word in (\'competition\',\'competitor\',\'competitive\',\'compete\',\'competing\') for suffix in (\'\',\'s\')}\nNEG_WORDS = {\'not\',\'less\',\'few\',\'limited\'}\nBLOCK_TAGS = {\'p\',\'div\',\'li\',\'td\',\'th\',\'tr\',\'pre\',\'h1\',\'h2\',\'h3\',\'h4\',\'h5\',\'h6\',\'section\',\'article\',\'blockquote\',\'br\',\'hr\'}\nC1_RE = re.compile(r\'[\\x80-\\x9f]\')\ntry:\n    import ftfy\nexcept ImportError:\n    ftfy=None\n\ndef repair_text(value):\n    value=value or \'\'\n    def c1(match):\n        try:return match.group().encode(\'latin1\').decode(\'cp1252\')\n        except UnicodeDecodeError:return \' \'\n    value=C1_RE.sub(c1,value)\n    if ftfy is not None:value=ftfy.fix_text(value,unescape_html=False)\n    return value.replace(\'\\xa0\',\' \').replace(\'\\u200b\',\'\')\n\ndef norm(value):return re.sub(r\'\\s+\',\' \',repair_text(value)).strip()\ndef words(value):return WORD_RE.findall(value)\ndef stable_hash(value):return hashlib.sha256(value.encode(\'utf-8\')).hexdigest()\n\ndef atom_json(path,data):\n    path=Path(path);temp=path.with_name(path.name+\'.tmp\')\n    temp.write_text(json.dumps(data,ensure_ascii=False,indent=2),encoding=\'utf-8\');temp.replace(path)\n\ndef score_text(text):\n    tokens=list(WORD_RE.finditer(text));low=[m.group().lower() for m in tokens];hits=[]\n    for i,m in enumerate(tokens):\n        if low[i] not in COMP_WORDS:continue\n        previous=low[max(0,i-3):i];excluded=[x for x in previous if x in NEG_WORDS]\n        hits.append({\'start\':m.start(),\'end\':m.end(),\'term\':m.group(),\'previous_3_words\':\' \'.join(previous),\n                     \'included\':not bool(excluded),\'excluded_by\':\'|\'.join(excluded)})\n    n=sum(h[\'included\'] for h in hits)\n    return {\'NWORDS\':len(tokens),\'NCOMP\':n,\'NCOMP_raw\':len(hits),\'NCOMP_excluded\':len(hits)-n,\n            \'PCTCOMP\':n/len(tokens) if tokens else None},hits\n\nassert score_text(\'Competition is strong.\')[0][\'NCOMP\']==1\nassert score_text(\'not a major competitor\')[0][\'NCOMP\']==0\nassert score_text(\'not a very large competitor\')[0][\'NCOMP\']==1\nassert score_text(\'Less competition; few competitors; limited competing options.\')[0][\'NCOMP\']==0\nassert score_text(\'Competition competitors competitive compete competing competitions competes\')[0][\'NCOMP\']==7\nassert score_text(\'competitiveness uncompetitive\')[0][\'NCOMP\']==0\nassert score_text(\'competition-like non-competition\')[0][\'NCOMP\']==2\nassert score_text(\'not. Competition\')[0][\'NCOMP\']==0 # Global token window, by explicit convention\nassert not C1_RE.search(repair_text(\'competitors\\x92 products and \\x93services\\x94\'))\n\n\n\ndef tagname(element):return str(element.tag).split(\'}\')[-1].split(\':\')[-1].lower()\n\ndef decode_html(raw):\n    source_bytes=raw\n    if raw.lstrip().upper().startswith(b\'<DOCUMENT\'):\n        begin=re.search(rb\'<html\\b\',raw,re.I);end=re.search(rb\'</html\\s*>\',raw,re.I)\n        if begin and end:raw=raw[begin.start():end.end()]\n    declaration=re.search(rb\'(?:charset\\s*=\\s*["\\\']?([\\w.-]+)|encoding\\s*=\\s*["\\\']([\\w.-]+))\',raw[:12000],re.I)\n    declared=next((x.decode(\'ascii\',\'ignore\') for x in declaration.groups() if x), \'\') if declaration else \'\'\n    if raw.startswith((b\'\\xff\\xfe\',b\'\\xfe\\xff\')):\n        text=raw.decode(\'utf-16\',errors=\'replace\');encoding=\'utf-16\'\n    else:\n        try:text=raw.decode(\'utf-8-sig\');encoding=\'utf-8\'\n        except UnicodeDecodeError:\n            candidate=declared.lower().replace(\'_\',\'-\')\n            if candidate in (\'\',\'ascii\',\'us-ascii\',\'iso-8859-1\',\'latin-1\',\'latin1\',\'windows-1252\'):candidate=\'cp1252\'\n            try:text=raw.decode(candidate,errors=\'strict\');encoding=candidate\n            except (UnicodeDecodeError,LookupError):text=raw.decode(\'cp1252\',errors=\'replace\');encoding=\'cp1252-with-replacements\'\n    text=re.sub(r\'^\\s*<\\?xml[^>]*\\?>\',\'\',text,flags=re.I)\n    return text,{\'source_encoding\':encoding,\'declared_encoding\':declared,\n                 \'decode_replacement_count\':text.count(\'\\ufffd\')}\n\nKNOWN_HEADINGS = {\n    \'general\',\'overview\',\'business overview\',\'business strategy\',\'our industry\',\'our history\',\n    \'business activities\',\'business segments\',\'competition\',\'competitors\',\n    \'market and competition\',\'competition and market served\',\'competition and markets served\',\n    \'competitive strengths\',\'our competitive strengths\',\'employees\',\'human capital\',\'human resources\',\n    \'available information\',\'internet information\',\'lending\',\'supervision and regulation\',\'regulation\',\n    \'government regulation\',\'research and development\',\'intellectual property\',\'properties\',\n    \'executive officers\',\'executive officers of the registrant\',\'our executive officers\',\n    \'customers\',\'suppliers\',\'backlog\',\'products\',\'technology\',\'seasonality\',\'position\',\'name\',\'age\'\n}\n\ndef has_speech(text):\n    return bool(re.search(r\'\\b(is|are|was|were|has|have|had|will|can|may|must|should|we|our|they|these|those|compete|competes|provide|provides|sell|sells|include|includes)\\b\',text,re.I))\n\ndef formatting_heading(element,text):\n    if len(text)>160 or len(words(text))>18:return False\n    if tagname(element) in {\'h1\',\'h2\',\'h3\',\'h4\',\'h5\',\'h6\'}:return True\n    styled=0;total=0\n    for node in element.iter():\n        if not isinstance(node.tag,str) or not node.text:continue\n        token=norm(node.text)\n        if not token:continue\n        total+=len(token)\n        chain=[node]+list(node.iterancestors())\n        strong=any(tagname(a) in {\'b\',\'strong\',\'i\',\'em\',\'u\'} or\n                   re.search(r\'(font-weight\\s*:\\s*(bold|[6-9]00)|font-style\\s*:\\s*italic|text-decoration\\s*:\\s*underline)\',a.get(\'style\',\'\'),re.I)\n                   for a in chain)\n        if strong:styled+=len(token)\n    return total>0 and styled/total>.8 and not has_speech(text) and not text.endswith((\';\',\',\'))\n\ndef classify_block(block):\n    text=block[\'text\'];low=text.lower().strip(\' .:;\\t\');n=len(words(text))\n    if not n:return \'numeric_or_symbol\',\'no_English_words\'\n    if block.get(\'personnel_table\'):return \'personnel_table\',\'structured_personnel_table\'\n    if n<=18 and low in KNOWN_HEADINGS:return \'heading\',\'recognized_heading_label\'\n    if block.get(\'style_heading\') or (n<=15 and len(text)<140 and text.isupper() and not has_speech(text)):\n        return \'heading\',\'short_formatted_heading\'\n    if block[\'in_table\'] and block.get(\'row_cells\',1)>1 and n<=8 and not has_speech(text) and not re.search(r\'[.!?;:]$\',text):\n        return \'table_label\',\'short_structured_table_cell\'\n    if block[\'tag\']==\'li\' or block.get(\'in_list\') or re.match(r\'^[\\u2022\\u00b7\\-]\\s*\',text):\n        return \'list_item\',\'\'\n    if block[\'in_table\']:return \'narrative_table_cell\',\'\'\n    return \'narrative\',\'\'\n\nKEEP_BLOCK_TYPES={\'narrative\',\'list_item\',\'narrative_table_cell\'}\n\ndef visible_blocks(raw):\n    decoded,encoding=decode_html(raw)\n    root=html.fromstring(decoded,parser=html.HTMLParser(recover=True,huge_tree=True))\n    xbrl={}\n    for e in root.xpath(\'//*[@name]\'):\n        name=e.get(\'name\',\'\')\n        if name.startswith(\'dei:\') and name not in xbrl:xbrl[name]=norm(e.text_content())\n    for e in list(root.iter()):\n        if not isinstance(e.tag,str):\n            if e.getparent() is not None:e.drop_tree()\n            continue\n        hidden=bool(re.search(r\'display\\s*:\\s*none|visibility\\s*:\\s*hidden\',e.get(\'style\',\'\'),re.I)) or \'hidden\' in e.attrib\n        if tagname(e) in {\'script\',\'style\',\'head\',\'noscript\',\'iframe\',\'object\'} or hidden or str(e.tag).lower().endswith(\':header\'):\n            if e.getparent() is not None:e.drop_tree()\n    personnel_tables=set()\n    for table in root.xpath(\'//table\'):\n        if table.xpath(\'.//table\'):continue  # Do not classify an outer layout table as personnel\n        text=norm(table.text_content()).lower()\n        roles=len(re.findall(r\'\\b(president|chief|officer|treasurer|secretary|director|chairman)\\b\',text))\n        header=bool(re.search(r\'\\bname\\b\',text) and re.search(r\'\\b(position|age|title|principal occupation)\\b\',text))\n        if len(text)<40000 and header and roles>=2:personnel_tables.add(table)\n    blocks=[];buffer=[];owner=[root]\n    def flush():\n        if not buffer:return\n        raw_text=\'\'.join(buffer);buffer.clear();text=norm(raw_text)\n        if not text:return\n        e=owner[0];ancestors=list(e.iterancestors());tables=[a for a in [e]+ancestors if tagname(a)==\'table\']\n        row=next((a for a in [e]+ancestors if tagname(a)==\'tr\'),None)\n        in_table=bool(tables)\n        block={\'text\':text,\'tag\':tagname(e),\n               \'style_heading\':formatting_heading(e,text),\'in_table\':in_table,\n               \'personnel_table\':any(t in personnel_tables for t in tables),\n               \'row_cells\':len(row.xpath(\'./td|./th\')) if row is not None else 1,\n               \'in_list\':any(tagname(a) in {\'li\',\'ul\',\'ol\'} for a in [e]+ancestors),\n               \n               }\n        block[\'block_type\'],block[\'exclusion_reason\']=classify_block(block)\n        if text.lower()==\'table of contents\':block.update(block_type=\'toc_header\',exclusion_reason=\'repeated_navigation_header\')\n        blocks.append(block)\n    def visit(e):\n        if not isinstance(e.tag,str):return\n        isblock=tagname(e) in BLOCK_TAGS;previous=owner[0]\n        if isblock:flush();owner[0]=e\n        if e.text:buffer.append(e.text)\n        for child in e:\n            visit(child)\n            if child.tail:buffer.append(child.tail)\n        if isblock:flush();owner[0]=previous\n    body=root.find(\'body\');visit(body if body is not None else root);flush()\n    # Retain numbered page blocks in the audit; pure numbers contribute zero English words.\n    # Cover metadata only; this does not add/remove visible words.\n    cover_registrants=[]\n    for table in root.xpath(\'//table\')[:30]:\n        header=norm(table.text_content())[:800].lower()\n        if not (\'registrant\' in header and \'commission\' in header and (\'employer\' in header or \'identification\' in header)):continue\n        for row in table.xpath(\'./tr|./tbody/tr\'):\n            cells=[norm(c.text_content()) for c in row.xpath(\'./td|./th\')]\n            if not cells or not re.fullmatch(r\'\\d{1,3}-\\d{3,6}\',cells[0]):continue\n            for value in cells[1:]:\n                if re.search(r\'\\b(company|corporation|corp\\.?|inc\\.?|limited|l\\.?p\\.?)\\b\',value,re.I) and len(value)<180:\n                    cover_registrants.append(value);break\n    xbrl[\'_cover_registrants\']=list(dict.fromkeys(cover_registrants))\n    refined=[]\n    pattern=re.compile(r\'^(\'+ \'|\'.join(re.escape(x) for x in sorted(KNOWN_HEADINGS,key=len,reverse=True)) +r\')([.:])\\s+(\\S.*)$\',re.I|re.S)\n    for block in blocks:\n        match=pattern.match(block[\'text\'])\n        if match and not block[\'personnel_table\'] and block[\'block_type\'] in KEEP_BLOCK_TYPES:\n            label=match.group(1)+match.group(2)\n            refined.append({**block,\'text\':label,\'block_type\':\'heading\',\'exclusion_reason\':\'inline_heading_prefix\',\'source_block_part\':0})\n            body={**block,\'text\':match.group(3),\'style_heading\':False,\'source_block_part\':1}\n            body[\'block_type\'],body[\'exclusion_reason\']=classify_block(body)\n            refined.append(body)\n        else:refined.append(block)\n    return refined,xbrl,encoding\n\n\nMARKER_RE = re.compile(r\'^item\\s*(1a|1b|1c|1|2)\\b\\s*[.:\\-–—]*\\s*(.*?)\\s*[.:\\-–—]*$\',re.I)\nCOMBINED_RE = re.compile(r\'^items?\\s*1\\s*\\.?\\s*(?:and|&)\\s*2\\s*\\.?\\s*[:\\-–—]?\\s*(.*?)\\s*[.:\\-–—]*$\',re.I)\nITEM_LABELS={\'1\':{\'\',\'business\',\'our business\',\'description of business\'},\n    \'1a\':{\'\',\'risk factors\',\'risk factors related to our business and operations\'},\n    \'1b\':{\'\',\'unresolved staff comments\'},\'1c\':{\'\',\'cybersecurity\'},\'2\':{\'\',\'properties\'}}\n\ndef item_markers(blocks):\n    found={}\n    for i in range(len(blocks)):\n        for width in (1,2,3):\n            part=blocks[i:i+width]\n            if len(part)!=width or any(len(b[\'text\'])>170 for b in part):continue\n            text=\' \'.join(b[\'text\'] for b in part)\n            if len(text)>220:continue\n            combined=COMBINED_RE.fullmatch(text)\n            match=MARKER_RE.fullmatch(text)\n            if combined:\n                num=\'1\';label=combined.group(1).lower().strip(\' .:-–—\')\n                if label not in {\'\',\'business and properties\',\'business & properties\'}:continue\n                scope=\'combined_item1_item2\'\n            elif match:\n                num,label=match.group(1).lower(),match.group(2).lower().strip(\' .:-–—\')\n                if label not in ITEM_LABELS[num]:continue\n                scope=\'item1_only\'\n            else:continue\n            candidate={\'index\':i,\'last\':i+width-1,\'item\':num,\'label\':label,\'heading\':text,\'scope\':scope}\n            old=found.get((i,num))\n            if old is None or (bool(label),width)>(bool(old[\'label\']),old[\'last\']-i+1):found[(i,num)]=candidate\n    # "Item 1." + "Business" may also be parsed as a marker on the next combined text;\n    # collapse only duplicate STARTS, not later body headings.\n    return sorted(found.values(),key=lambda x:(x[\'index\'],x[\'last\']))\n\ndef locate_item1(blocks):\n    markers=item_markers(blocks);candidates=[]\n    for st in (m for m in markers if m[\'item\']==\'1\'):\n        ends=[m for m in markers if m[\'index\']>st[\'last\'] and m[\'item\']!=\'1\']\n        if not ends:continue\n        en=ends[0]\n        if any(m[\'item\']==\'1\' and st[\'last\']<m[\'index\']<en[\'index\'] for m in markers):continue\n        a,b=st[\'last\']+1,en[\'index\']\n        nw=sum(len(words(x[\'text\'])) for x in blocks[a:b])\n        if nw<100:continue\n        score=nw+(200 if st[\'label\'] else 0)+(100 if en[\'item\']==\'1a\' else 0)\n        candidates.append((score,a,b,st,en,nw))\n    if not candidates:\n        return None,{\'extract_status\':\'review_missing_item1\',\'candidate_headings\':[m[\'heading\'] for m in markers[:30]],\n                     \'item1_scope\':\'unresolved\',\'item1_scope_review_required\':True}\n    _,a,b,st,en,nw=max(candidates,key=lambda x:x[0])\n    status=\'ok\' if en[\'item\']==\'1a\' and nw>=300 and st[\'label\'] else \'review_boundary\'\n    return (a,b),{\'extract_status\':status,\'item1_start_block\':a,\'item1_end_block_exclusive\':b,\n        \'item1_start_heading\':st[\'heading\'],\'item1_end_heading\':en[\'heading\'],\n        \'item1_visible_words\':nw,\'candidate_heading_count\':len(markers),\n        \'item1_scope\':st[\'scope\'],\'item1_scope_review_required\':st[\'scope\']!=\'item1_only\'}\n\nMONTHS=\'January February March April May June July August September October November December\'\nDATE_RE=re.compile(r\'(?i)(?:\'+MONTHS.replace(\' \',\'|\')+r\')\\s+\\d{1,2},?\\s+(?:19|20)\\d{2}|(?:19|20)\\d{2}[-/]\\d{1,2}[-/]\\d{1,2}|\\d{1,2}/\\d{1,2}/(?:19|20)\\d{2}\')\nOVERRIDES={}\nif METADATA_OVERRIDES_CSV is not None:\n    overrides_frame=pd.read_csv(METADATA_OVERRIDES_CSV,dtype=str).fillna(\'\')\n    if \'accession\' not in overrides_frame:raise ValueError(\'Metadata overrides require an accession column\')\n    if overrides_frame.accession.duplicated().any():raise ValueError(\'Duplicate accession in metadata overrides\')\n    OVERRIDES=overrides_frame.set_index(\'accession\').to_dict(\'index\')\n\ndef date_key(text):\n    try:return pd.to_datetime(text).date().isoformat()\n    except Exception:return re.sub(r\'[^a-z0-9]\',\'\',text.lower())\n\n# Audited source-specific corrections. These are accession+CIK keyed, NOT timeless CIK-name mappings.\n# Other joint reports remain unresolved until their issuer evidence is supplied/available.\nAUDITED_ISSUER_CORRECTIONS={\n (\'0001193125-10-040857\',\'0000003146\'):{\n  \'registrant\':\'Alabama Gas Corporation\',\n  \'evidence\':\'https://www.sec.gov/Archives/edgar/data/277595/000119312509168492/0001193125-09-168492-index.htm\',\n  \'note\':\'CIK and EIN verified in SEC index; historical name confirmed on uploaded 2009 report cover.\'},\n (\'0000092122-10-000009\',\'0000003153\'):{\n  \'registrant\':\'Alabama Power Company\',\n  \'evidence\':\'https://www.sec.gov/Archives/edgar/data/92122/000009212210000009/0000092122-10-000009-index.htm\',\n  \'note\':\'CIK verified in this filing index; full company name confirmed on uploaded report cover.\'}}\n\ndef plausible_registrant(value):\n    value=norm(value).lstrip(\'([ \')\n    return bool(2<=len(value)<=180 and re.search(\'[A-Za-z]\',value) and\n        not re.search(r\'(?i)^(?:commission|file number|state of|registrant\\b|exact name|annual report|for the |securities |irs employer|i\\.r\\.s)\',value))\n\ndef registrant_from_source(source,xbrl,full_text,raw,blocks):\n    match=re.match(r\'^(\\d+)_(\\d{10}-\\d{2}-\\d{6})_\',source.name)\n    cik=match.group(1).zfill(10) if match else str(xbrl.get(\'dei:EntityCentralIndexKey\',\'\')).zfill(10)\n    accession=match.group(2) if match else \'\'\n    names=xbrl.get(\'_cover_registrants\',[])\n    joint=len(names)>1\n    evidence=\'\';candidate=\'\';method=\'\'\n    if raw:\n        # A complete SEC submission header may contain several filers. Match issuer CIK explicitly.\n        header=raw[:max(0,raw.upper().find(b\'<DOCUMENT>\'))].decode(\'utf-8\',errors=\'replace\') if b\'<SEC-HEADER>\' in raw.upper() else \'\'\n        for section in re.split(r\'(?i)\\bFILER\\s*:\',header)[1:]:\n            cm=re.search(r\'CENTRAL INDEX KEY\\s*:\\s*(\\d+)\',section,re.I)\n            nm=re.search(r\'COMPANY CONFORMED NAME\\s*:\\s*([^\\r\\n]+)\',section,re.I)\n            if cm and nm and cm.group(1).zfill(10)==cik:\n                candidate=norm(nm.group(1));method=\'SEC_header_matching_issuer_CIK\';break\n    correction=AUDITED_ISSUER_CORRECTIONS.get((accession,cik))\n    if not candidate and correction and correction[\'registrant\'] in names:\n        candidate=correction[\'registrant\'];method=\'audited_accession_CIK_cover_mapping\';evidence=correction[\'evidence\']\n    if not candidate:\n        xname=xbrl.get(\'dei:EntityRegistrantName\',\'\')\n        xcik=str(xbrl.get(\'dei:EntityCentralIndexKey\',\'\')).zfill(10)\n        if plausible_registrant(xname) and (not joint or xcik==cik):candidate=xname;method=\'XBRL_matching_issuer\' if joint else \'XBRL\'\n    if not candidate and not joint:\n        patterns=[(r\'COMPANY CONFORMED NAME\\s*:\\s*([^\\n]+)\',\'SEC_header_single_issuer\'),\n          (r\'([^\\n]{2,160})\\n\\n\\(?exact name of registrant\',\'cover_exact_name\'),\n          (r\'commission\\s+file\\s+(?:number|no\\.?)\\s*[^\\n]*\\n\\n([^\\n]{2,150})\',\'cover_after_file_number\')]\n        for pattern,label in patterns:\n            for found in re.finditer(pattern,full_text[:40000],re.I):\n                if plausible_registrant(found.group(1)):\n                    candidate=norm(found.group(1));method=label;break\n            if candidate:break\n    if not candidate and not joint and len(names)==1:candidate=names[0];method=\'single_cover_registrant\'\n    if not candidate and not joint and blocks:\n        boundary,_=locate_item1(blocks)\n        if boundary:\n            for block in blocks[boundary[0]:min(boundary[0]+8,boundary[1])]:\n                # Image-only covers: strict corporate-name prefix in the opening business narrative.\n                found=re.match(r\'^(.{2,110}?\\b(?:Inc\\.|Corporation|Company|Corp\\.|Limited|LLC))(?:\\s|[,\\(])\',block[\'text\'],re.I)\n                if found and plausible_registrant(found.group(1)):\n                    candidate=found.group(1);method=\'opening_business_name_prefix\';evidence=block[\'text\'][:300];break\n    return candidate,method,evidence,names,joint\n\n\ndef filing_metadata(source,xbrl,full_text,raw=None,blocks=None):\n    m=re.match(r\'^(\\d+)_(\\d{10}-\\d{2}-\\d{6})_\',source.name)\n    cik=(m.group(1) if m else xbrl.get(\'dei:EntityCentralIndexKey\',\'\')).lstrip(\'0\') or \'\'\n    accession=m.group(2) if m else \'\'\n    registrant=registrant_source=registrant_evidence=\'\';cover_names=xbrl.get(\'_cover_registrants\',[]);joint=len(cover_names)>1\n    period=xbrl.get(\'dei:DocumentPeriodEndDate\',\'\')\n    if not period:\n        match=re.search(r\'(?:fiscal\\s+)?year\\s+ended\\s*[:\\-]?\\s*(.{0,160})\',full_text[:30000],re.I|re.S)\n        date=DATE_RE.search(match.group(1)) if match else None\n        period=norm(date.group()) if date else \'\'\n    try:period_date=pd.to_datetime(period).date() if period else None\n    except Exception:period_date=None\n    fy=xbrl.get(\'dei:DocumentFiscalYearFocus\',\'\')\n    fy_source=\'dei:DocumentFiscalYearFocus\' if re.fullmatch(r\'(?:19|20)\\d{2}\',fy) else \'\'\n    evidence=\'\';review=False;candidate_year=None\n    if not fy_source and period_date and period_date.month==12 and period_date.day==31:\n        fy=str(period_date.year);fy_source=\'calendar_Dec31_year\'\n    if not fy_source:\n        fy=\'\'\n        patterns=[\n            r\'all\\s+references\\s+to\\s+((?:19|20)\\d{2}).{0,150}?(?:mean|refer to).{0,70}?fiscal\\s+years?\\s+ended\\s*([^.;]{0,200})\',\n            r\'fiscal(?:\\s+year)?\\s+((?:19|20)\\d{2})\\b(.{0,200})\',\n            r\'((?:19|20)\\d{2})\\s+fiscal\\s+year\\b(.{0,200})\']\n        for pattern_index,pattern in enumerate(patterns):\n            for match in re.finditer(pattern,full_text[:120000],re.I|re.S):\n                date=DATE_RE.search(match.group(2))\n                if date and period and (pattern_index==0 or re.search(r\'\\bend(?:ed|ing)?\\b\',match.group(2)[:date.start()],re.I)) and date_key(date.group())==date_key(period):\n                    fy=match.group(1);fy_source=\'explicit_fiscal_year_date_mapping\';evidence=norm(match.group())[:300];break\n            if fy:break\n        if not fy and period_date:\n            candidate_year=period_date.year-1 if period_date.month==1 and period_date.day<=15 else period_date.year\n            if period_date.month==12 and period_date.day==31:\n                fy=str(period_date.year);fy_source=\'calendar_Dec31_year\'\n            elif period_date.month!=1:\n                # Report the convention transparently instead of leaving ordinary noncalendar FYs blank.\n                fy=str(period_date.year);fy_source=\'inferred_period_end_year\';review=True\n            else:\n                fy_source=\'unresolved_January_fiscal_label\';review=True\n    override=OVERRIDES.get(accession,{})\n    if override.get(\'fiscal_year\'):\n        fy=override[\'fiscal_year\'];fy_source=\'user_metadata_override\';review=False;evidence=override.get(\'note\',\'\')\n    if override.get(\'registrant\'):registrant=override[\'registrant\'];registrant_source=\'user_metadata_override\'\n    fiscal_year=int(fy) if re.fullmatch(r\'(?:19|20)\\d{2}\',str(fy)) else None\n    source_year=getattr(source,\'filing_year\',\'\')\n    if not source_year:\n        source_year=next((part for part in reversed(getattr(source,\'parts\',())[:-1]) if re.fullmatch(r\'(?:19|20)\\d{2}\',part)), \'\')\n    accession_year=2000+int(accession.split(\'-\')[1]) if accession else None\n    filing_year=int(source_year) if source_year else accession_year\n    form=xbrl.get(\'dei:DocumentType\',\'\')\n    if not form and raw:\n        match=re.search(rb\'<TYPE>\\s*([^\\r\\n<]+)\',raw[:10000],re.I)\n        form=match.group(1).decode(\'ascii\',\'ignore\').strip() if match else \'\'\n    return {\'cik\':cik.zfill(10) if cik else \'\', \'accession\':accession,\'registrant\':registrant,\n            \'registrant_source\':registrant_source,\'registrant_evidence\':registrant_evidence,\n            \'cover_registrants\':cover_names,\'is_joint_report\':joint,\n            \'document_scope\':\'joint_registrant_report\' if joint else \'single_or_unresolved_issuer\',\n            \'issuer_name_resolved\':bool(registrant),\'issuer_scope_review_required\':joint,\'form_type\':form,\'fiscal_year\':fiscal_year,\n            \'fiscal_year_candidate\':candidate_year,\'fiscal_year_source\':fy_source,\n            \'fiscal_year_review_required\':review or fiscal_year is None,\'fiscal_year_evidence\':evidence,\n            \'period_end\':period_date.isoformat() if period_date else period,\n            \'period_end_year\':period_date.year if period_date else None,\'filing_year\':filing_year,\n            \'source_quarter\':getattr(source,\'quarter\',\'\'),\n            \'source_year_mismatch\':bool(source_year and accession_year and int(source_year)!=accession_year),\n            \'metadata_review_required\':not bool(cik and fiscal_year and registrant) or review or joint}\n\n\nfrom zipfile import ZipFile, is_zipfile\nfrom dataclasses import dataclass\n\n_ZIP_READERS={}  # one open archive at a time; avoid parsing its central directory for every filing\n\n@dataclass\nclass Source:\n    file: Path\n    member: str = \'\'\n    filing_year: str = \'\'\n    quarter: str = \'\'\n    @property\n    def name(self):return Path(self.member).name if self.member else self.file.name\n    @property\n    def parts(self):return self.file.parts\n    def resolve(self):return str(self.file.resolve())+(\'!\'+self.member if self.member else \'\')\n    def __str__(self):return self.resolve()\n    def read_bytes(self):\n        if self.member:\n            if self.file not in _ZIP_READERS:\n                for old in _ZIP_READERS.values():old.close()\n                _ZIP_READERS.clear();_ZIP_READERS[self.file]=ZipFile(self.file)\n            return _ZIP_READERS[self.file].read(self.member)\n        return self.file.read_bytes()\n\n\n\n# The builder prepends the verified text, block-parser, metadata and Source definitions.\nfrom functools import lru_cache\n\n@lru_cache(maxsize=20000)\ndef cached_compound(text):\n    return float(sentiment.polarity_scores(text)[\'compound\'])\n\ndef initialize_worker(settings):\n    import spacy\n    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer\n    globals().update(settings)\n    globals()[\'ROOT\']=Path(settings[\'ROOT\']);globals()[\'CACHE\']=Path(settings[\'CACHE\'])\n    globals()[\'nlp\']=spacy.blank(\'en\');nlp.add_pipe(\'sentencizer\');nlp.max_length=10000000\n    globals()[\'sentiment\']=SentimentIntensityAnalyzer()\n    feature=stable_hash(json.dumps({\'spacy\':metadata.version(\'spacy\'),\'vaderSentiment\':metadata.version(\'vaderSentiment\'),\n        \'ftfy\':metadata.version(\'ftfy\') if ftfy is not None else \'fallback_c1_repair\',\n        \'cleaner_version\':CLEANER_VERSION,\'lm_dictionary_sha256\':None},sort_keys=True))[:16]\n    globals()[\'FEATURE_TAG\']=feature\n    globals()[\'LEGACY_INDEX\']={p.name:str(p) for directory in settings[\'LEGACY_CACHE_DIRS\']\n        if Path(directory).is_dir() for p in Path(directory).glob(\'*.json.gz\')}\n\n\ndef is_review(filing,location,encoding_issue=False):\n    reasons=[]\n    if not filing.get(\'cik\'):reasons.append(\'missing_CIK\')\n    if filing.get(\'fiscal_year\') is None or filing.get(\'fiscal_year_review_required\'):reasons.append(\'fiscal_year_review\')\n    if filing.get(\'is_joint_report\'):reasons.append(\'joint_report\')\n    if location.get(\'item1_scope\')==\'combined_item1_item2\':reasons.append(\'combined_Items_1_2\')\n    if encoding_issue:reasons.append(\'remaining_encoding_issue\')\n    return reasons\n\n\ndef select_filing_fields(filing,review):\n    return {key:filing.get(key) for key in (\'doc_id\',\'cik\',\'fiscal_year\',\'item1_sentiment\')} | {\n        \'review_required\':bool(review),\'review_reasons\':review}\n\n\ndef convert_legacy(record):\n    old=record[\'filing\']\n    review=is_review(old,old,any(p.get(\'paragraph_encoding_issue\') for p in record[\'paragraphs\']))\n    paragraphs=[{key:p[key] for key in (\'paragraph_id\',\'paragraph_text\',\'paragraph_competition_sentence_count\',\n        \'para_word_count\',\'para_avg_sentence_words\',\'para_avg_word_chars\',\'para_type_token_ratio\')}\n        for p in record[\'paragraphs\']]\n    sentences=[{key:s[key] for key in (\'paragraph_id\',\'sentence_id\',\'sentence_text\',\'competition_binary\')}\n        for s in record[\'sentences\']]\n    return {\'filing\':select_filing_fields(old,review),\'paragraphs\':paragraphs,\'sentences\':sentences}\n\n\ndef extract_native(source,raw,doc_id):\n    blocks,xbrl,encoding=visible_blocks(raw)\n    boundary,location=locate_item1(blocks)\n    if not boundary:return None,location[\'extract_status\']\n    if location[\'extract_status\']!=\'ok\':return None,location[\'extract_status\']\n    full=\'\\n\\n\'.join(\'\' if b[\'block_type\']==\'toc_header\' else b[\'text\'] for b in blocks)\n    filing=filing_metadata(source,xbrl,full,raw,blocks=blocks);filing[\'doc_id\']=doc_id\n    del full\n    a,b=boundary\n    # Preserve the original FULL-DOCUMENT previous-three-word convention.\n    history=[]\n    for block in reversed(blocks[:a]):\n        if block[\'block_type\']==\'toc_header\':continue\n        history=words(block[\'text\'])+history\n        if len(history)>=3:break\n    history=[w.lower() for w in history[-3:]]\n    candidates=[];encoding_issue=False\n    for index in range(a,b):\n        block=blocks[index]\n        if block[\'block_type\']==\'toc_header\':continue\n        text=block[\'text\'];tokens=list(WORD_RE.finditer(text));valid_positions=[]\n        for token in tokens:\n            word=token.group().lower()\n            if word in COMP_WORDS and not any(w in NEG_WORDS for w in history):valid_positions.append(token.start())\n            history=(history+[word])[-3:]\n        if block[\'block_type\'] not in KEEP_BLOCK_TYPES:continue\n        if C1_RE.search(text) or \'\\ufffd\' in text:encoding_issue=True\n        candidates.append((index,text,[m.group() for m in tokens],valid_positions))\n    paragraphs=[];sentences=[];score_sum=0.;score_count=0\n    # Batch sentence segmentation; do not start a separate spaCy process pool inside each worker.\n    docs=nlp.pipe((item[1] for item in candidates),batch_size=128,n_process=1)\n    for (index,text,ws,positions),doc in zip(candidates,docs):\n        pid=f\'{doc_id}:p{index:06d}\';units=[s for s in doc.sents if s.text.strip()]\n        positives=0;assigned=0\n        for number,unit in enumerate(units,1):\n            count=bisect.bisect_left(positions,unit.end_char)-bisect.bisect_left(positions,unit.start_char)\n            assigned+=count\n            binary=int(count>0);positives+=binary;sentence=unit.text.strip()\n            sentences.append({\'paragraph_id\':pid,\'sentence_id\':f\'{pid}:s{number:04d}\',\n                              \'sentence_text\':sentence,\'competition_binary\':binary})\n            score_sum+=cached_compound(sentence);score_count+=1\n        assert assigned==len(positions),\'Sentence coverage missed a target occurrence\'\n        if not units:continue\n        n=len(ws);low=[w.lower() for w in ws]\n        paragraphs.append({\'paragraph_id\':pid,\'paragraph_text\':text,\n            \'paragraph_competition_sentence_count\':positives,\'para_word_count\':n,\n            \'para_avg_sentence_words\':n/len(units),\'para_avg_word_chars\':sum(map(len,ws))/max(1,n),\n            \'para_type_token_ratio\':len(set(low))/max(1,n)})\n    if not paragraphs:return None,\'review_no_narrative\'\n    filing[\'item1_sentiment\']=score_sum/score_count if score_count else None\n    review=is_review(filing,location,encoding_issue)\n    return {\'filing\':select_filing_fields(filing,review),\'paragraphs\':paragraphs,\'sentences\':sentences},\'ok\'\n\n\ndef process_spec(spec):\n    started=time.perf_counter();source=Source(Path(spec[\'file\']),spec[\'member\'],spec[\'filing_year\'],spec[\'quarter\'])\n    cache_key=stable_hash(spec[\'stamp\']+\'|\'+LITE_VERSION+\'|\'+FEATURE_TAG)[:32]\n    cache=CACHE/(cache_key+\'.json.gz\')\n    base={\'source_path\':str(source),\'source_stamp\':spec[\'stamp\']}\n    try:\n        if cache.exists():\n            with gzip.open(cache,\'rt\',encoding=\'utf-8\') as handle:record=json.load(handle)\n            origin=\'lean_cache\'\n        else:\n            raw=source.read_bytes();raw_hash=hashlib.sha256(raw).hexdigest()\n            match=re.match(r\'^(\\d+)_(\\d{10}-\\d{2}-\\d{6})_\',source.name)\n            accession=match.group(2) if match else \'\'\n            issuer=match.group(1).zfill(10) if match else source.name\n            doc_id=stable_hash(issuer+\'|\'+(accession or source.name)+\'|\'+raw_hash)[:24]\n            context=json.dumps({\'parser\':PARSER_VERSION,\'rule\':RULE_VERSION,\'features\':FEATURE_TAG,\'overrides\':{},\n                \'audited_issuers\':{(\'|\'.join(k)):v for k,v in AUDITED_ISSUER_CORRECTIONS.items()}},sort_keys=True)\n            old_key=stable_hash(doc_id+\'|\'+context)[:32]+\'.json.gz\';legacy=LEGACY_INDEX.get(old_key)\n            record=None\n            if legacy:\n                with gzip.open(legacy,\'rt\',encoding=\'utf-8\') as handle:old=json.load(handle)\n                f=old[\'filing\']\n                if f.get(\'source_sha256\')==raw_hash and f.get(\'extract_status\')==\'ok\' and old[\'paragraphs\']:\n                    record=convert_legacy(old)\n                del old\n            if record is not None:origin=\'v2_cache\'\n            else:\n                record,status=extract_native(source,raw,doc_id);origin=\'new_extraction\'\n                if record is None:return {**base,\'status\':status,\'cache_origin\':origin,\'seconds\':time.perf_counter()-started}\n            temporary=cache.with_name(cache.name+f\'.{os.getpid()}.tmp\')\n            with gzip.open(temporary,\'wt\',encoding=\'utf-8\',compresslevel=1) as handle:\n                json.dump(record,handle,ensure_ascii=False,separators=(\',\',\':\'))\n            temporary.replace(cache)\n        filing=record[\'filing\']\n        return {**base,**filing,\'status\':\'ok\',\'cache_path\':str(cache),\'cache_origin\':origin,\n                \'sentence_rows\':len(record[\'sentences\']),\'paragraphs\':len(record[\'paragraphs\']),\n                \'seconds\':time.perf_counter()-started}\n    except Exception as exc:\n        return {**base,\'status\':\'error\',\'error\':f\'{type(exc).__name__}: {exc}\',\'seconds\':time.perf_counter()-started}\n\n\ndef run_jobs(settings_path,jobs_path,workers):\n    from concurrent.futures import ProcessPoolExecutor\n    from collections import deque\n    import multiprocessing as mp\n    settings=json.loads(Path(settings_path).read_text(encoding=\'utf-8\'))\n    def specs():\n        with Path(jobs_path).open(encoding=\'utf-8\') as handle:\n            for line in handle:yield json.loads(line)\n    jobs=iter(specs())\n    if workers==1:\n        initialize_worker(settings)\n        for spec in jobs:print(\'RESULT_JSON \'+json.dumps(process_spec(spec),ensure_ascii=False),flush=True)\n        return\n    pool=ProcessPoolExecutor(max_workers=workers,mp_context=mp.get_context(\'spawn\'),\n                             initializer=initialize_worker,initargs=(settings,))\n    pending=deque()\n    try:\n        # Bounded prefetch preserves source order and deterministic LDA sampling/partitions.\n        # At most 2*workers futures exist, rather than tens of thousands queued at once.\n        for _ in range(workers*2):\n            spec=next(jobs,None)\n            if spec is None:break\n            pending.append(pool.submit(process_spec,spec))\n        while pending:\n            result=pending.popleft().result()\n            print(\'RESULT_JSON \'+json.dumps(result,ensure_ascii=False),flush=True)\n            spec=next(jobs,None)\n            if spec is not None:pending.append(pool.submit(process_spec,spec))\n    finally:\n        for future in pending:future.cancel()\n        pool.shutdown(wait=True,cancel_futures=True)\n\nif __name__==\'__main__\':\n    import argparse,signal\n    parser=argparse.ArgumentParser()\n    parser.add_argument(\'--settings\',required=True);parser.add_argument(\'--jobs\',required=True)\n    parser.add_argument(\'--workers\',type=int,required=True)\n    args=parser.parse_args()\n    if args.workers<1:raise ValueError(\'workers must be >=1\')\n    def stop_requested(*unused):raise KeyboardInterrupt(\'Notebook requested stop\')\n    signal.signal(signal.SIGTERM,stop_requested)\n    run_jobs(args.settings,args.jobs,args.workers)\n'

# RUNNER_SOURCE is embedded by this Notebook; no separate script download is needed.
def worker_results(jobs):
    settings={'ROOT':str(ROOT),'CACHE':str(CACHE),'LEGACY_CACHE_DIRS':LEGACY_CACHE_DIRS,
        'LITE_VERSION':LITE_VERSION,'PARSER_VERSION':PARSER_VERSION,'RULE_VERSION':RULE_VERSION,'CLEANER_VERSION':CLEANER_VERSION}
    config_path=RUNTIME/'settings.json';atom_json(config_path,settings)
    jobs_path=RUNTIME/'jobs.jsonl'
    with jobs_path.open('w',encoding='utf-8') as handle:
        for spec in jobs:handle.write(json.dumps(spec,ensure_ascii=False)+'\n')
    script=RUNTIME/('lean_runner_'+stable_hash(RUNNER_SOURCE)[:12]+'.py');script.write_text(RUNNER_SOURCE,encoding='utf-8')
    environment=os.environ.copy()
    for name in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS'):environment[name]='1'
    environment['TOKENIZERS_PARALLELISM']='false'
    error_path=RUNTIME/'worker_errors.log'
    with error_path.open('w',encoding='utf-8') as errors:
        process=subprocess.Popen([sys.executable,'-u',str(script),'--settings',str(config_path),'--jobs',str(jobs_path),
            '--workers',str(CPU_WORKERS)],stdout=subprocess.PIPE,stderr=errors,text=True,encoding='utf-8',bufsize=1,
            env=environment,start_new_session=(os.name=='posix'))
        try:
            for line in process.stdout:
                if line.startswith('RESULT_JSON '):yield json.loads(line[len('RESULT_JSON '):])
            code=process.wait()
            if code:raise RuntimeError(f'Worker run stopped ({code}). Set CPU_WORKERS=1 if necessary.\n'+error_path.read_text(errors='replace')[-2500:])
        finally:
            if process.poll() is None:
                try:
                    if os.name=='posix':os.killpg(process.pid,signal.SIGINT)
                    else:process.terminate()
                    process.wait(timeout=15)
                except (ProcessLookupError,subprocess.TimeoutExpired):
                    if process.poll() is None:
                        if os.name=='posix':os.killpg(process.pid,signal.SIGTERM)
                        else:process.kill()
                        try:process.wait(timeout=5)
                        except subprocess.TimeoutExpired:process.kill();process.wait()
            if process.stdout:process.stdout.close()


## 5. Shared LDA and lightweight export functions

There is ONE model for the whole selected run, not one per company, quarter or Excel part. A term-count matrix is built from complete retained business-section narrative and lists, using English stopwords plus company boilerplate terms. Training is sampled across available years/quarters; all other accepted filings are transformed with the frozen model. Missing vectors are left blank and review-flagged, not filled with zeroes.

Each workbook has one `AllSentences` sheet. CSV, if enabled, uses the same schema. Each part is finalized at approximately 25,000 sentence rows; ordinary paragraphs stay together, while a report may span parts, so you can open the first completed part before the full run ends. Excel-overlength cells are logged separately; complete paragraph/sentence text remains in the compact cache. Read the CURRENT `export_manifest.json` rather than wildcard-concatenating stale files from a previous larger run.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer,ENGLISH_STOP_WORDS
from sklearn.decomposition import LatentDirichletAllocation
import joblib
from openpyxl import Workbook
from openpyxl.cell import WriteOnlyCell
from openpyxl.styles import Font,PatternFill
from openpyxl.writer.excel import ExcelWriter
from zipfile import ZIP_DEFLATED

model_bundle=None;model_info={};empty_topic_docs=[]

def fit_shared_model(training):
    global model_bundle,model_info
    if LDA_MODEL_TO_REUSE is not None:
        model_bundle=joblib.load(LDA_MODEL_TO_REUSE)
        if not {'vectorizer','lda'}<=set(model_bundle):raise ValueError('Model bundle needs vectorizer and lda keys')
        model_info={'status':'reused_explicit_model','model_file':str(LDA_MODEL_TO_REUSE)}
        model_path=Path(LDA_MODEL_TO_REUSE)
    elif len(training)>=5:
        stops=sorted(set(ENGLISH_STOP_WORDS)|{'inc','corp','corporation','company','companies','fiscal','www','com'})
        vectorizer=CountVectorizer(stop_words=stops,min_df=2,max_df=.98,max_features=30000,
                                  token_pattern=r'(?u)\b[a-zA-Z][a-zA-Z\-]{2,}\b')
        try:X=vectorizer.fit_transform(item1_text(ref) for ref in training)
        except ValueError as exc:
            model_info={'status':'no_usable_training_vocabulary','reason':str(exc)};return
        topics=min(LDA_MAX_TOPICS,max(2,len(training)//50),X.shape[1])
        lda=LatentDirichletAllocation(n_components=topics,max_iter=LDA_ITERATIONS,
            learning_method='batch',random_state=42,n_jobs=1)
        lda.fit(X);del X
        model_bundle={'vectorizer':vectorizer,'lda':lda,'training_doc_ids':[r['doc_id'] for r in training],
            'parser_version':PARSER_VERSION,'feature_scope':'retained_business_section_narrative',
            'exploratory_small_sample':len(training)<50*topics}
        model_path=MODEL_DIR/'shared_lda.joblib';joblib.dump(model_bundle,model_path)
        model_info={'status':'fitted','model_file':str(model_path),'n_training_documents':len(training),
            'exploratory_small_sample':model_bundle['exploratory_small_sample']}
    else:
        model_info={'status':'fewer_than_5_training_filings','n_training_documents':len(training)};return
    model_info.update(model_id=hashlib.sha256(model_path.read_bytes()).hexdigest()[:16],
                      topics=int(model_bundle['lda'].n_components))
    terms=model_bundle['vectorizer'].get_feature_names_out()
    pd.DataFrame([{'topic':i+1,'top_words':' '.join(terms[np.argsort(row)[-15:][::-1]])}
        for i,row in enumerate(model_bundle['lda'].components_)]).to_csv(MODEL_DIR/'topic_terms.csv',index=False)
    print('Shared LDA ready:',model_info)


def fit_and_record(training):
    fit_shared_model(training)
    atom_json(MODEL_DIR/'model_info.json',model_info)

def topic_vectors(records):
    if model_bundle is None:return [None]*len(records)
    text=['\n\n'.join(p['paragraph_text'] for p in rec['paragraphs']) for rec in records]
    X=model_bundle['vectorizer'].transform(text);vectors=model_bundle['lda'].transform(X)
    empty=np.asarray(X.sum(axis=1)).ravel()==0
    return [None if is_empty else vector.tolist() for vector,is_empty in zip(vectors,empty)]


INVALID_XML=re.compile(r'[\x00-\x08\x0b\x0c\x0e-\x1f]')

class MainWriter:
    def __init__(self):
        self.part=0;self.rows=0;self.total=0;self.book=None;self.csv_handle=None;self.manifest=[];self.overflow=0
        self.overflow_handle=(OUT/'excel_overflow.csv').open('w',encoding='utf-8-sig',newline='')
        self.overflow_writer=csv.writer(self.overflow_handle)
        self.overflow_writer.writerow(['sentence_id','column','full_characters'])
        atom_json(OUT/'export_manifest.json',{'columns':COLS,'parts':[],'completed_sentence_rows':0})
    def open_part(self):
        self.part+=1;self.rows=0
        if EXPORT_EXCEL:
            self.book=Workbook(write_only=True);self.sheet=self.book.create_sheet('AllSentences');self.sheet.freeze_panes='C2'
            self.sheet.column_dimensions['A'].width=70;self.sheet.column_dimensions['B'].width=70
            header=[]
            for value in COLS:
                cell=WriteOnlyCell(self.sheet,value=value);cell.font=Font(bold=True,color='FFFFFF')
                cell.fill=PatternFill('solid',fgColor='17365D');header.append(cell)
            self.sheet.append(header)
        if EXPORT_CSV:
            self.csv_path=EXPORT_DIR/f'competition_part{self.part:04d}.csv'
            self.csv_temp=self.csv_path.with_suffix('.tmp.csv')
            self.csv_handle=self.csv_temp.open('w',encoding='utf-8-sig',newline='')
            self.csv_writer=csv.writer(self.csv_handle);self.csv_writer.writerow(COLS)
    def reserve_paragraph(self,count):
        if self.rows and self.rows+count>EXCEL_ROWS_PER_FILE:self.finish_part()
    def add(self,row):
        if not self.rows and self.book is None and self.csv_handle is None:self.open_part()
        if self.rows>=EXCEL_ROWS_PER_FILE:self.finish_part();self.open_part()
        values=[row.get(key) for key in COLS]
        if self.csv_handle:self.csv_writer.writerow(values)
        if self.book:
            cells=[]
            for key,value in zip(COLS,values):
                if isinstance(value,str):
                    value=INVALID_XML.sub('',value)
                    if len(value)>32767:
                        self.overflow+=1;self.overflow_writer.writerow([row['sentence_id'],key,len(value)])
                        value=value[:32670]+' [SHORTENED IN EXCEL; full text remains in cache]'
                    cell=WriteOnlyCell(self.sheet,value=value);cell.data_type='s';cells.append(cell)
                else:cells.append(value)
            self.sheet.append(cells)
        self.rows+=1;self.total+=1
    def finish_part(self):
        if not self.rows:return
        paths=[]
        if self.csv_handle:
            self.csv_handle.close();self.csv_temp.replace(self.csv_path);paths.append(str(self.csv_path));self.csv_handle=None
        if self.book:
            path=EXPORT_DIR/f'competition_part{self.part:04d}.xlsx';temporary=path.with_suffix('.tmp.xlsx')
            # Fast ZIP compression; cell values are identical to a normal XLSX save.
            with ZipFile(temporary,'w',ZIP_DEFLATED,compresslevel=1,allowZip64=True) as archive:
                ExcelWriter(self.book,archive).save()
            temporary.replace(path);paths.append(str(path));self.book=None
        self.manifest.append({'part':self.part,'sentence_rows':self.rows,'paths':paths})
        atom_json(OUT/'export_manifest.json',{'columns':COLS,'parts':self.manifest,'completed_sentence_rows':self.total})
        print('Saved:',', '.join(paths),'| rows:',self.rows,flush=True)
        self.rows=0
    def close(self):
        self.finish_part();self.overflow_handle.close()


def export_batch(refs,writer):
    records=[read_record(ref) for ref in refs];vectors=topic_vectors(records)
    for ref,record,vector in zip(refs,records,vectors):
        filing=record['filing'];paragraphs={p['paragraph_id']:p for p in record['paragraphs']}
        if vector is None:empty_topic_docs.append(ref['doc_id'])
        topic_json=json.dumps(vector,separators=(',',':')) if vector is not None else None
        positive=Counter(s['paragraph_id'] for s in record['sentences'] if s['competition_binary'])
        for pid,p in paragraphs.items():assert positive[pid]==p['paragraph_competition_sentence_count']
        by_paragraph={pid:[] for pid in paragraphs}
        for sentence in record['sentences']:by_paragraph[sentence['paragraph_id']].append(sentence)
        for pid,paragraph in paragraphs.items():
            writer.reserve_paragraph(len(by_paragraph[pid]))
            for sentence in by_paragraph[pid]:
                assert sentence['competition_binary'] in (0,1)
                row={**filing,**paragraph,**sentence,'item1_lda_topic_probabilities':topic_json,
                     'review_required':bool(filing['review_required'] or vector is None)}
                writer.add(row)


## 6. Run — extraction, model fit and early exports

Run this cell. The initial sampled sources are extracted first; then LDA fits and exports begin while remaining extraction continues. The first-item ETA includes startup and is unreliable; judge throughput after at least 100 sources. Cold start still requires processing the training sample before the first complete Excel file appears.

Wait for a Saved message before opening a workbook. Interrupt stops the isolated worker process group on Mac/Linux and finalizes rows already passed to the writer. Compact per-filing caches remain reusable, including files extracted before model fitting. To resume, keep the settings and rerun steps 2–6. Do not run two writers against this output folder simultaneously. If source coverage or model settings change, all current exports are rebuilt with a newly shared model; completed extraction caches are reused.

The tiny `run_log.jsonl` stores source/exception/review information instead of repeating it across the main Excel. An interrupted run has `complete=false` in `run_summary.json`. Check this flag and coverage before describing the collection as complete.


In [ ]:
writer=MainWriter();training=[];buffer=[];seen=set();statuses=Counter();origins=Counter();accepted=0
model_ready=False;started=time.perf_counter();finished=False
log_path=OUT/'run_log.jsonl';cache_manifest=OUT/'cache_manifest.jsonl'
atom_json(OUT/'run_summary.json',{'complete':False,'state':'in_progress','selected_sources':len(ordered_jobs),'columns':COLS})
iterator=worker_results(ordered_jobs)
if training_jobs==0:
    fit_and_record([]);model_ready=True
try:
    with log_path.open('w',encoding='utf-8') as log,cache_manifest.open('w',encoding='utf-8') as manifest:
        for completed,result in enumerate(tqdm(iterator,total=len(ordered_jobs),desc='CPU extraction + export'),1):
            status=result['status'];statuses[status]+=1;origins[result.get('cache_origin','unknown')]+=1
            result['duplicate']=False
            if status=='ok':
                duplicate=result['doc_id'] in seen;result['duplicate']=duplicate
                if not duplicate:
                    seen.add(result['doc_id']);accepted+=1
                    manifest.write(json.dumps(result,ensure_ascii=False)+'\n');manifest.flush()
                    if model_ready:buffer.append(result)
                    else:training.append(result)
            log.write(json.dumps(result,ensure_ascii=False)+'\n');log.flush()
            if not model_ready and completed==training_jobs:
                fit_and_record(training);model_ready=True;buffer.extend(training);training.clear()
                while len(buffer)>=EXPORT_BATCH_FILINGS:
                    batch=buffer[:EXPORT_BATCH_FILINGS];del buffer[:EXPORT_BATCH_FILINGS];export_batch(batch,writer)
            if model_ready and len(buffer)>=EXPORT_BATCH_FILINGS:
                export_batch(buffer,writer);buffer.clear()
        if not model_ready:
            fit_and_record(training);buffer.extend(training);training.clear()
        if buffer:export_batch(buffer,writer);buffer.clear()
        finished=True
finally:
    iterator.close()
    writer.close()  # Finalize completed rows even if you interrupted later extraction.
    model_info['documents_without_topic_vector']=empty_topic_docs
    atom_json(MODEL_DIR/'model_info.json',model_info)
    atom_json(OUT/'run_summary.json',{'complete':finished,'selected_sources':len(ordered_jobs),'accepted_filings':accepted,
        'sentence_rows_exported':writer.total,'statuses':dict(statuses),'cache_origins':dict(origins),
        'workers':CPU_WORKERS,'seconds':time.perf_counter()-started,'columns':COLS,'excel_overflow_cells':writer.overflow,
        'note':'Read only files listed in export_manifest.json. The initial source sample trains ONE shared model.'})
print('Finished:',finished,'| accepted filings:',accepted,'| exported sentences:',writer.total)
print('Cache reuse:',dict(origins));print('Data:',EXPORT_DIR)


## The 16 main columns

| Column | Meaning / calculation |
|---|---|
| paragraph_text | Complete retained paragraph/list text; repeated for its sentence rows. |
| sentence_text | One spaCy sentence/list unit from that paragraph. |
| competition_binary | 1 if this unit contains any included dictionary occurrence, otherwise 0. |
| paragraph_competition_sentence_count | Number of positive units in this paragraph, repeated on each child row. |
| item1_lda_topic_probabilities | JSON list of document-level topic probabilities under the ONE saved model; sums to 1 when available. |
| item1_sentiment | Unweighted mean VADER compound score over ALL retained units in the report business section. Generic sentiment baseline, not validated financial tone. |
| cik | Issuer CIK as a zero-padded string; used for later company matching. |
| fiscal_year | Report fiscal-year label; uncertain/missing labels trigger review. Not source collection year. |
| doc_id | Content/issuer-based report identifier; distinct cofilers of a shared report remain separate. |
| paragraph_id | Stable parent-paragraph ID for the same source and parser, including later embedding joins. |
| sentence_id | Stable sentence-unit ID within that paragraph. |
| para_word_count | Number of [A-Za-z]+ English word tokens in the paragraph. |
| para_avg_sentence_words | Paragraph word count / sentence-unit count. |
| para_avg_word_chars | Total letters in paragraph word tokens / paragraph word count. |
| para_type_token_ratio | Number of distinct lowercase word tokens / word count. |
| review_required | True if FY/CIK is unresolved/inferred, a detected joint report or combined Item1/2 section is used, encoding remains suspect, or a topic vector is unavailable. Reasons stay in the small logs. |

Text descriptors are candidates for length/style controls, not validated writing-quality scores or automatically valid causal controls. Keep all 0 and 1 rows. Report/paragraph features repeat across child sentence rows: aggregate using doc_id/paragraph_id rather than summing repeated values. Combined Item1/2 and multi-registrant text remain explicitly review-flagged. ROI and embedding can later be joined using the IDs; no embedding placeholder column is included.

## Verification limits

The underlying v2.1 parser was checked on the eleven available uploaded HTML sources. This lean implementation is checked against v2-style extraction for text, sentence labels, paragraph counts and the retained metrics; spawn workers, ZIP input, shared-report IDs, old/new cache reuse, early XLSX/CSV exports and fixed model application are tested locally. The authoring runtime lacks spaCy/VADER, so local structural tests use explicit TEST-ONLY stand-ins; the delivered code imports the actual packages. No Mac/full-corpus throughput benchmark or GPU run is claimed. Start with a small scope if changing the environment. No universal fastest setting or causal validity is guaranteed by the software.
